# Fine-tuning Qwen2.5-Coder-7B for Code Review Comments

QLoRA fine-tune of `unsloth/Qwen2.5-Coder-7B-Instruct` on real GitHub PR review comments, so it learns to leave the kind of comment a senior engineer would — including knowing when to say nothing.

**Before running:** `Runtime` -> `Change runtime type` -> **T4 GPU**.

This notebook is self-contained: it downloads the dataset, formats it, trains, and saves the adapter. For the full quantitative evaluation (baseline vs. fine-tuned, LLM-as-judge) and for serving the model, use `evaluate.py` and `serve.py` from the project repo after downloading the saved adapter folder.


## 1. Install dependencies


In [ ]:
import os

if "COLAB_" not in "".join(os.environ.keys()):
    # Not on Colab (e.g. local or Kaggle) -- plain install
    %pip install -q unsloth
else:
    # Colab-pinned install order avoids CUDA/dependency conflicts
    %pip install -q --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft \
        trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    %pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
    %pip install -q --no-deps unsloth


## 2. Load the base model in 4-bit


In [ ]:
import torch
from unsloth import FastLanguageModel

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Go to Runtime > Change runtime type > T4 GPU, then re-run."
    )
print("GPU:", torch.cuda.get_device_name(0))

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,          # auto-detect bf16/fp16 support
    load_in_4bit=True,
)


## 3. Prepare the dataset

[`ronantakizawa/github-codereview`](https://huggingface.co/datasets/ronantakizawa/github-codereview): 167K+ real `(before_code, reviewer_comment, after_code)` triplets from 725 top GitHub repos, plus ~51K "clean code, no comment needed" negatives. We filter to Python, apply a quality threshold, and keep ~15% negatives so the model learns restraint, not just fault-finding.


In [ ]:
import json
import random

from datasets import Dataset, load_dataset

random.seed(42)

LANGUAGE = "Python"
MIN_QUALITY = 0.3
MAX_TRAIN = 6000       # keep this modest so a full run fits a free T4 session
NEGATIVE_FRAC = 0.15

SYSTEM_PROMPT = (
    "You are an expert code reviewer. You will be shown a chunk of code "
    "from a pull request, with its language and file path. Write the "
    "single most important review comment a senior engineer would leave. "
    "Be specific and actionable. If the code has no issues worth "
    "flagging, respond with exactly: No issues found."
)


def build_example(row):
    user_msg = (
        f"File: {row['file_path']}\n"
        f"Language: {row['language']}\n\n"
        f"Diff:\n{row['diff_context']}\n\n"
        f"Code chunk:\n{row['before_code']}"
    )
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": row["reviewer_comment"].strip()},
        ]
    }


raw = load_dataset("ronantakizawa/github-codereview")

splits = {}
for split, cap in [("train", MAX_TRAIN), ("validation", 400), ("test", 500)]:
    rows = raw[split].filter(lambda r: r["language"] == LANGUAGE)
    pos = [r for r in rows if not r["is_negative"] and r["quality_score"] >= MIN_QUALITY]
    neg = [r for r in rows if r["is_negative"]]
    random.shuffle(pos)
    random.shuffle(neg)

    n_neg = int(cap * NEGATIVE_FRAC)
    n_pos = cap - n_neg
    chosen = pos[:n_pos] + neg[:n_neg]
    random.shuffle(chosen)

    splits[split] = [build_example(r) for r in chosen]
    print(f"{split}: {len(splits[split])} examples ({n_neg} negative)")

# Also save to disk so evaluate.py can reuse the exact same test set later
import pathlib
pathlib.Path("data").mkdir(exist_ok=True)
for split, fname in [("train", "train.jsonl"), ("validation", "val.jsonl"), ("test", "test.jsonl")]:
    with open(f"data/{fname}", "w") as f:
        for ex in splits[split]:
            f.write(json.dumps(ex) + "\n")


## 4. Configure LoRA


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


## 5. Format for training

Render each example through Qwen's chat template so special tokens match what the base model was pretrained with.


In [ ]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}


train_ds = Dataset.from_list(splits["train"]).map(formatting_prompts_func, batched=True)
val_ds = Dataset.from_list(splits["validation"]).map(formatting_prompts_func, batched=True)


## 6. Set up the trainer

`train_on_responses_only` masks the loss to the assistant's tokens only -- without it, the model spends gradient steps learning to predict the prompt it was just given, which wastes capacity and is the detail most from-scratch SFT setups get wrong.


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


## 7. Train

Roughly 45-90 minutes on a free T4 with the defaults above, depending on how many examples you kept.


In [ ]:
trainer_stats = trainer.train()


## 8. Save the adapter


In [ ]:
model.save_pretrained("qwen_codereview_lora")
tokenizer.save_pretrained("qwen_codereview_lora")

print("Saved to ./qwen_codereview_lora -- download this folder before the runtime recycles.")

# Optional: push straight to the Hub instead of downloading manually
# from huggingface_hub import login
# login()  # paste a HF token with write access when prompted
# model.push_to_hub("YOUR_USERNAME/qwen2.5-coder-7b-codereview-lora")
# tokenizer.push_to_hub("YOUR_USERNAME/qwen2.5-coder-7b-codereview-lora")


## 9. Quick sanity check

A few qualitative examples before you move to the full harness in `evaluate.py`.


In [ ]:
FastLanguageModel.for_inference(model)

for ex in splits["validation"][:3]:
    msgs = ex["messages"][:2]  # system + user only -- let the model generate the reply
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("=" * 80)
    print("MODEL:         ", generated.strip())
    print("HUMAN REVIEWER:", ex["messages"][2]["content"])


## Next steps

1. Download the `qwen_codereview_lora` folder (or confirm the Hub push worked).
2. Run `evaluate.py` on a GPU machine for the full baseline-vs-fine-tuned comparison, including the LLM-as-judge scoring.
3. Run `serve.py --serve` to stand up a local endpoint, or `--code path/to/file.py` for a one-off review.
